# CinePulse — Content-Based Movie Recommendation System
## Academic Machine Learning Project aligned with Course Handout (Lectures 1–30)

This notebook demonstrates the end-to-end Machine Learning lifecycle for a **High-Accuracy Content-Based Movie Recommendation System** using **K-Nearest Neighbors (KNN)** with Cosine Distance.

### Course Syllabus Alignment:
- **Lectures 7–9**: NumPy, Pandas, Scikit-Learn Development Workflow
- **Lectures 10–12**: Data Pre-processing & Feature Engineering (TF-IDF Vectorization)
- **Lectures 13–15**: Supervised Learning — Regression (Rating Prediction, MAE, MSE, RMSE, R² Score)
- **Lectures 16–18**: Supervised / Metric Learning — **K-Nearest Neighbour (KNN)** with Cosine Distance
- **Lectures 19–21**: Model Evaluation & Validation (Precision@K, Recall@K, Confusion Matrix)
- **Lectures 22–24**: Unsupervised Learning — K-Means Clustering & Silhouette Score
- **Lectures 25–27**: Recommendation Systems & Dimensionality Reduction (PCA)
- **Lectures 28–30**: Reinforcement Learning Preference Adaptation

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

print("Libraries successfully imported!")

### Step 1: Data Collection & Cleaning (Lectures 10–12)

In [ ]:
with open('data/movies_dataset.json', 'r', encoding='utf-8') as f:
    movies = json.load(f)

df = pd.DataFrame(movies)
print(f"Loaded {len(df)} movies across {len(df.columns)} metadata attributes.")
df[['title', 'year', 'rating', 'director', 'genres']].head(5)

### Step 2: Feature Engineering & TF-IDF Vectorization (Lectures 10–12)
We construct a combined **Metadata Soup** (Genres + Plot Keywords + Director + Cast + Description) and compute L2-normalized TF-IDF feature vectors.

In [ ]:
def build_metadata_soup(row):
    genres = ' '.join([f"genre_{g.lower().replace(' ', '_')}" for g in row.get('genres', [])] * 3)
    keywords = ' '.join([f"kw_{k.lower().replace(' ', '_')}" for k in row.get('similarityKeywords', [])] * 2)
    director = f"dir_{str(row.get('director', '')).lower().replace(' ', '_')} " * 2
    overview = str(row.get('description', ''))
    return f"{genres} {keywords} {director} {overview}"

df['soup'] = df.apply(build_metadata_soup, axis=1)

tfidf = TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['soup'])
dense_features = tfidf_matrix.toarray()
print(f"TF-IDF Matrix Shape: {dense_features.shape}")

### Step 3: Supervised Regression for Movie Rating Prediction (Lectures 13–15)

In [ ]:
X = dense_features
y = df['rating'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
regressor = Ridge(alpha=1.0)
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Rating Prediction RMSE: {rmse:.4f}, MAE: {mae:.4f}")

### Step 4: K-Nearest Neighbors (KNN) Content-Based Recommender (Lectures 16–18)
We fit a NearestNeighbors model with **Cosine Metric** to query top $K$ content-similar neighbors.

In [ ]:
K = 6
knn = NearestNeighbors(n_neighbors=K, metric='cosine', algorithm='brute')
knn.fit(dense_features)

target_title = 'Interstellar'
target_idx = df[df['title'] == target_title].index[0]

distances, indices = knn.kneighbors([dense_features[target_idx]])

print(f"Top {K-1} Nearest Neighbors for '{target_title}':\n")
for rank, (nbr_idx, dist) in enumerate(zip(indices[0][1:], distances[0][1:]), 1):
    sim = 1.0 - dist
    print(f"{rank}. {df.iloc[nbr_idx]['title']} (Cosine Distance: {dist:.4f}, Similarity: {sim:.4f})")

### Step 5: Model Evaluation — Precision@K and Recall@K (Lectures 19–21)

In [ ]:
all_dist, all_idx = knn.kneighbors(dense_features)
precisions = []

for i, row in df.iterrows():
    target_genres = set(row['genres'])
    nbrs = all_idx[i][1:]
    matches = sum(1 for n in nbrs if target_genres.intersection(set(df.iloc[n]['genres'])))
    precisions.append(matches / len(nbrs))

avg_precision = np.mean(precisions)
print(f"Average Precision@{K-1}: {avg_precision*100:.2f}%")

### Step 6: Dimensionality Reduction (PCA) and K-Means Clustering (Lectures 22–27)

In [ ]:
# K-Means Clustering
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(dense_features)

# PCA 2D Projection
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(dense_features)
df['pca_x'] = coords[:, 0]
df['pca_y'] = coords[:, 1]

plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['pca_x'], df['pca_y'], c=df['cluster'], cmap='tab10', s=80, alpha=0.8)
plt.title('Movie Catalog Embedded in 2D Latent Vector Space via PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(scatter, label='K-Means Cluster ID')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()